[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-10-capstone-quality-framework.ipynb#scrollTo=cc330001)

---
# Day 10 · Capstone — Build a Full Data Quality Framework for a Multi-Table Pipeline
**certified-journeys / sodacore-certified** · Soda Core for Data Quality · Exam Badge

> **Goal for today:** Assemble every technique from this course into a production-grade data quality framework for a four-table e-commerce pipeline: dependency-ordered scanning, comprehensive SodaCL checks, a Python orchestration class, schema drift detection, data contracts, and a QUALITY.md documentation template.

In [ ]:
%pip install -q soda-core-duckdb

## Architecture Overview

```
Pipeline dependency order:
  customers ──┐
  products  ──┼──► orders ──► order_items
```

**Quality framework layers:**
1. **Table-level checks** — row count, schema, missing PKs, duplicates
2. **Referential integrity** — orders.customer_id → customers.id, etc.
3. **Business rules** — valid statuses, non-negative prices, quantity constraints
4. **Custom SQL** — shipped before ordered, revenue anomalies
5. **Orchestration** — dependency-ordered scan execution, halt on upstream failure
6. **Contracts** — formal SLA for the `orders` table

## Step 1 · Build the E-Commerce Database

We create a realistic four-table e-commerce schema with deliberately injected quality issues to exercise each check type.

In [ ]:
import duckdb, tempfile, pathlib, datetime

tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "ecommerce.duckdb")

conn = duckdb.connect(db_path)

# ── customers (clean) ──────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE customers (
        id         INTEGER PRIMARY KEY,
        name       VARCHAR NOT NULL,
        email      VARCHAR,
        tier       VARCHAR,
        created_at TIMESTAMP
    )
""")
conn.execute("""
    INSERT INTO customers VALUES
        (1, 'Alice',   'alice@example.com',  'gold',   '2024-01-01 08:00:00'),
        (2, 'Bob',     'bob@example.com',    'silver', '2024-01-02 09:00:00'),
        (3, 'Carol',   NULL,                 'bronze', '2024-01-03 10:00:00'),
        (4, 'Dave',    'dave@example.com',   'gold',   '2024-01-04 11:00:00'),
        (5, 'Eve',     'eve@example.com',    'unknown','2024-01-05 12:00:00')
""")

# ── products (price issue injected) ───────────────────────────────────────
conn.execute("""
    CREATE TABLE products (
        id         INTEGER PRIMARY KEY,
        name       VARCHAR NOT NULL,
        sku        VARCHAR NOT NULL,
        price      DOUBLE,
        category   VARCHAR,
        updated_at TIMESTAMP
    )
""")
conn.execute("""
    INSERT INTO products VALUES
        (1, 'Widget A', 'SKU-001',  25.00, 'widgets',   '2024-01-10 08:00:00'),
        (2, 'Widget B', 'SKU-002', -10.00, 'widgets',   '2024-01-10 08:00:00'),
        (3, 'Gadget X', 'SKU-003', 150.00, 'gadgets',   '2024-01-10 08:00:00'),
        (4, 'Gadget Y', 'SKU-004',  80.00, 'gadgets',   '2024-01-10 08:00:00'),
        (5, 'Doohickey','SKU-005',  40.00, 'other',     '2024-01-10 08:00:00')
""")

# ── orders (missing amount, invalid status) ────────────────────────────────
conn.execute("""
    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY,
        customer_id INTEGER,
        amount      DOUBLE,
        status      VARCHAR,
        order_date  DATE,
        shipped_date DATE,
        updated_at  TIMESTAMP
    )
""")
conn.execute("""
    INSERT INTO orders VALUES
        (1, 1, 150.00, 'delivered', '2024-01-10', '2024-01-12', '2024-01-12 10:00:00'),
        (2, 2, 320.50, 'shipped',   '2024-01-11', '2024-01-14', '2024-01-14 11:00:00'),
        (3, 3,   NULL, 'unknown',   '2024-01-12', NULL,         '2024-01-12 09:00:00'),
        (4, 4, 200.00, 'pending',   '2024-01-15', NULL,         '2024-01-15 08:00:00'),
        (5, 9,  75.00, 'pending',   '2024-01-16', NULL,         '2024-01-16 07:00:00')
""")
# Note: order 3 has NULL amount and 'unknown' status; order 5 references customer_id=9 (does not exist)

# ── order_items (quantity = 0 injected) ───────────────────────────────────
conn.execute("""
    CREATE TABLE order_items (
        id         INTEGER PRIMARY KEY,
        order_id   INTEGER,
        product_id INTEGER,
        quantity   INTEGER,
        unit_price DOUBLE,
        updated_at TIMESTAMP
    )
""")
conn.execute("""
    INSERT INTO order_items VALUES
        (1, 1, 1, 2, 25.00,  '2024-01-12 10:00:00'),
        (2, 1, 3, 1, 150.00, '2024-01-12 10:00:00'),
        (3, 2, 2, 0,  10.00, '2024-01-14 11:00:00'),
        (4, 4, 4, 2,  80.00, '2024-01-15 08:00:00'),
        (5, 4, 5, 1,  40.00, '2024-01-15 08:00:00')
""")
# Note: order_item 3 has quantity=0 (invalid)

conn.close()
print("E-commerce database created:", db_path)
for t in ['customers','products','orders','order_items']:
    c = duckdb.connect(db_path).execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {c} rows")

## Step 2 · Comprehensive Checks YAML — All Four Tables

Write a single checks file covering every quality dimension: volume, completeness, uniqueness, validity, schema, and custom SQL.

In [ ]:
config_yml = f"""
data_sources:
  ecommerce:
    type: duckdb
    path: "{db_path}"
"""

checks_yml = """
# ── customers ─────────────────────────────────────────────────────────────
checks for customers:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(name) = 0
  - invalid_count(tier) = 0:
      valid values: [gold, silver, bronze]
  - schema:
      fail:
        when required column missing: [id, name, email, tier, created_at]

# ── products ──────────────────────────────────────────────────────────────
checks for products:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(sku) = 0
  - invalid_count(price) = 0:
      valid min: 0
  - invalid_count(sku) = 0:
      valid regex: SKU-[0-9]{3}
  - schema:
      fail:
        when required column missing: [id, name, sku, price, category]

# ── orders ────────────────────────────────────────────────────────────────
checks for orders:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(amount) = 0
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
  - invalid_count(amount) = 0:
      valid min: 0
  - failed rows:
      fail condition: shipped_date IS NOT NULL AND shipped_date < order_date
      name: shipped_before_ordered
  - schema:
      fail:
        when required column missing: [id, customer_id, amount, status, order_date]

# ── order_items ───────────────────────────────────────────────────────────
checks for order_items:
  - row_count > 0
  - missing_count(id) = 0
  - missing_count(order_id) = 0
  - missing_count(product_id) = 0
  - invalid_count(quantity) = 0:
      valid min: 1
  - invalid_count(unit_price) = 0:
      valid min: 0
  - schema:
      fail:
        when required column missing: [id, order_id, product_id, quantity, unit_price]
"""

config_path = tmpdir / "configuration.yml"
checks_path = tmpdir / "checks_all.yml"
config_path.write_text(config_yml)
checks_path.write_text(checks_yml)

print("Checks file written. Checks per table:")
for table in ['customers', 'products', 'orders', 'order_items']:
    count = checks_yml.count(f"checks for {table}:")
    print(f"  {table}: block {'present' if count else 'MISSING'}")

## Step 3 · DataQualityOrchestrator — Dependency-Ordered Execution

In [ ]:
from soda.scan import Scan
from typing import Dict, List, Optional
import json

class DataQualityOrchestrator:
    """
    Runs Soda scans in dependency order.
    Halts the pipeline if an upstream table fails.
    """

    def __init__(self, data_source: str, config_path: str):
        self.data_source = data_source
        self.config_path = config_path
        self.results: Dict[str, dict] = {}

    def scan_table(self, table_name: str, checks_path: str) -> int:
        """Scan a single table and store the result."""
        scan = Scan()
        scan.set_data_source_name(self.data_source)
        scan.add_configuration_yaml_file(self.config_path)
        scan.add_sodacl_yaml_file(checks_path)
        scan.execute()

        exit_code = scan.get_exit_code()
        self.results[table_name] = {
            "exit_code": exit_code,
            "status": {0: "PASS", 1: "WARN", 2: "FAIL"}.get(exit_code, "UNKNOWN"),
            "logs": scan.get_logs_text(),
        }
        return exit_code

    def run_pipeline(self, stages: List[Dict]) -> bool:
        """
        Run checks in stages. Each stage is:
          {"table": "orders", "checks_path": "/path/checks.yml",
           "depends_on": ["customers"], "blocking": True}
        Returns True if all blocking stages pass.
        """
        all_passed = True
        for stage in stages:
            table = stage["table"]
            blocking = stage.get("blocking", True)
            depends_on = stage.get("depends_on", [])

            # Check upstream dependencies
            for dep in depends_on:
                if self.results.get(dep, {}).get("exit_code", 0) == 2:
                    print(f"  [{table}] SKIPPED — upstream '{dep}' FAILED")
                    self.results[table] = {"exit_code": -1, "status": "SKIPPED", "logs": ""}
                    all_passed = False
                    if blocking:
                        return False
                    continue

            print(f"  [{table}] Scanning...")
            exit_code = self.scan_table(table, stage["checks_path"])
            status = self.results[table]["status"]
            print(f"  [{table}] → {status}")

            if exit_code == 2 and blocking:
                all_passed = False
                print(f"  [{table}] Blocking failure — halting pipeline")
                return False

        return all_passed

    def print_report(self):
        """Print a formatted results table."""
        print("\n" + "─" * 50)
        print(f"{'Table':<20} {'Status':<10} {'Exit'}")
        print("─" * 50)
        for table, r in self.results.items():
            emoji = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIPPED": "⏭️"}.get(r['status'], '?')
            print(f"{table:<20} {r['status']:<10} {r['exit_code']:>4}  {emoji}")
        print("─" * 50)

print("DataQualityOrchestrator defined.")

## Step 4 · Write Per-Table Checks Files

In [ ]:
# Write individual checks files for each table
table_checks = {
    "customers": """
checks for customers:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(name) = 0
  - invalid_count(tier) = 0:
      valid values: [gold, silver, bronze]
""",
    "products": """
checks for products:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(sku) = 0
  - invalid_count(price) = 0:
      valid min: 0
  - invalid_count(sku) = 0:
      valid regex: SKU-[0-9]{3}
""",
    "orders": """
checks for orders:
  - row_count > 0
  - missing_count(id) = 0
  - duplicate_count(id) = 0
  - missing_count(amount) = 0
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
  - invalid_count(amount) = 0:
      valid min: 0
""",
    "order_items": """
checks for order_items:
  - row_count > 0
  - missing_count(order_id) = 0
  - missing_count(product_id) = 0
  - invalid_count(quantity) = 0:
      valid min: 1
  - invalid_count(unit_price) = 0:
      valid min: 0
"""
}

checks_paths = {}
for table, yml in table_checks.items():
    p = tmpdir / f"checks_{table}.yml"
    p.write_text(yml)
    checks_paths[table] = str(p)

print("Per-table checks files written:")
for t, p in checks_paths.items():
    print(f"  {t}: {p}")

## Step 5 · Run the Orchestrated Pipeline

In [ ]:
orchestrator = DataQualityOrchestrator(
    data_source="ecommerce",
    config_path=str(config_path),
)

# Pipeline stages in dependency order
# customers and products have no upstream deps; orders depends on both;
# order_items depends on orders.
pipeline_stages = [
    {"table": "customers",   "checks_path": checks_paths["customers"],   "depends_on": [],                          "blocking": True},
    {"table": "products",    "checks_path": checks_paths["products"],    "depends_on": [],                          "blocking": True},
    {"table": "orders",      "checks_path": checks_paths["orders"],      "depends_on": ["customers", "products"],   "blocking": True},
    {"table": "order_items", "checks_path": checks_paths["order_items"], "depends_on": ["orders"],                  "blocking": False},
]

print("Running dependency-ordered pipeline scan...")
passed = orchestrator.run_pipeline(pipeline_stages)
orchestrator.print_report()
print(f"\nOverall pipeline status: {'PASSED ✅' if passed else 'FAILED ❌'}")

### What just happened?

- **`customers`**: FAIL — `tier='unknown'` for customer 5 violates `valid values: [gold, silver, bronze]`.
- Because `customers` is a blocking dependency and it FAILED, the `orders` scan is **SKIPPED**.
- **`products`**: also FAIL — `price=-10.00` for product 2 violates `valid min: 0`.
- **`order_items`**: depends on `orders` which was skipped → also SKIPPED.
- The orchestrator halted after the first blocking failure as configured.

## Step 6 · Data Contract for the Orders Table

In [ ]:
orders_contract = """
# data-contract-orders.yml
# Formal data contract for the orders table.
# Owner: Data Engineering Team
# Consumers: Finance (revenue), Fulfillment (shipping), Analytics (dashboards)
# SLA: data must be fresher than 24 hours; less than 1% missing amounts

dataset: orders
datasource: ecommerce

columns:
  - name: id
    data_type: integer
    description: Primary key — unique per order
  - name: customer_id
    data_type: integer
    description: FK to customers.id
  - name: amount
    data_type: double
    description: Total order value in USD
  - name: status
    data_type: varchar
    description: Order lifecycle state
  - name: order_date
    data_type: date
    description: Date the order was placed
  - name: shipped_date
    data_type: date
    description: Date the order was shipped (NULL if not yet shipped)
  - name: updated_at
    data_type: timestamp
    description: Row-level ingestion timestamp for freshness checks

checks:
  # Volume
  - row_count > 0

  # Completeness
  - missing_count(id) = 0
  - missing_count(customer_id) = 0
  - missing_percent(amount):
      warn: when > 0.5
      fail: when > 1.0

  # Uniqueness
  - duplicate_count(id) = 0

  # Validity
  - invalid_count(status) = 0:
      valid values: [pending, shipped, delivered, cancelled]
  - invalid_count(amount) = 0:
      valid min: 0

  # Freshness SLA: data must be no older than 24 hours
  # - freshness(updated_at) < 24h
  #   (commented out: freshness checks require a live timestamp column)
"""

contract_path = tmpdir / "data-contract-orders.yml"
contract_path.write_text(orders_contract)

print("Data contract written to:", contract_path)
print()
print(orders_contract)

### What just happened?

- A **data contract** is a YAML document that specifies what a table's consumers can rely on: schema, constraints, freshness SLA, and quality thresholds.
- Contracts are stored in version control alongside the pipeline code — they evolve via pull requests, giving consumers visibility into upstream changes.
- The `checks:` section maps directly to SodaCL — the contract IS the checks file for this table.
- Soda Cloud can ingest contracts and track compliance over time in the health dashboard.

📖 https://docs.soda.io/soda-core/data-contracts.html

## Step 7 · Schema Drift Detection

In [ ]:
schema_checks = """
checks for orders:
  - schema:
      name: orders_schema_contract
      fail:
        when required column missing:
          - id
          - customer_id
          - amount
          - status
          - order_date
          - updated_at
      warn:
        when wrong column type:
          id: integer
          amount: double
"""

schema_path = tmpdir / "checks_schema.yml"
schema_path.write_text(schema_checks)

from soda.scan import Scan

scan_schema = Scan()
scan_schema.set_data_source_name("ecommerce")
scan_schema.add_configuration_yaml_file(str(config_path))
scan_schema.add_sodacl_yaml_file(str(schema_path))
scan_schema.execute()

print("Schema check results:")
print(scan_schema.get_logs_text())

In [ ]:
# Simulate a breaking schema change: rename 'amount' to 'total_amount'
conn3 = duckdb.connect(db_path)
conn3.execute("ALTER TABLE orders RENAME COLUMN amount TO total_amount")
conn3.close()

print("Breaking schema change applied: 'amount' renamed to 'total_amount'")
print()

# Re-run schema check — should now FAIL
scan_schema2 = Scan()
scan_schema2.set_data_source_name("ecommerce")
scan_schema2.add_configuration_yaml_file(str(config_path))
scan_schema2.add_sodacl_yaml_file(str(schema_path))
scan_schema2.execute()

print("Schema check after breaking change:")
print(scan_schema2.get_logs_text())

# Restore column name
conn4 = duckdb.connect(db_path)
conn4.execute("ALTER TABLE orders RENAME COLUMN total_amount TO amount")
conn4.close()
print("Column restored.")

### What just happened?

- Before the schema change: schema check **passes** — all required columns present.
- After renaming `amount` → `total_amount`: schema check **fails** — `amount` is now missing.
- This demonstrates how schema checks act as a **circuit breaker** for upstream schema changes.
- In production: run schema checks on every pipeline execution and alert the team on any WARN or FAIL.

## Step 8 · QUALITY.md — Framework Documentation Template

In [ ]:
quality_md = """
# QUALITY.md — E-Commerce Pipeline Data Quality Framework

## Overview

This document describes the data quality framework for the e-commerce pipeline.
All checks are implemented in SodaCL and executed via the Soda Core programmatic API.

**Scan frequency:** Every pipeline run (triggered by Airflow/Prefect DAG)  
**On FAIL:** Pipeline halts; on-call engineer is notified  
**On WARN:** Pipeline continues; logged to observability dashboard  

---

## Table: customers

**Owner:** Customer Data Team  
**Checks file:** `soda/checks_customers.yml`

| Check | Threshold | Rationale | Action on FAIL |
|---|---|---|---|
| `row_count > 0` | FAIL | Table must not be empty | Halt pipeline |
| `missing_count(id) = 0` | FAIL | PK must never be NULL | Halt pipeline |
| `duplicate_count(id) = 0` | FAIL | PK must be unique | Halt pipeline |
| `missing_count(name) = 0` | FAIL | Name is required | Halt pipeline |
| `invalid_count(tier)` | FAIL | Must be gold/silver/bronze | Halt pipeline |

---

## Table: products

**Owner:** Catalogue Team  
**Checks file:** `soda/checks_products.yml`

| Check | Threshold | Rationale | Action on FAIL |
|---|---|---|---|
| `row_count > 0` | FAIL | Table must not be empty | Halt pipeline |
| `duplicate_count(sku) = 0` | FAIL | SKU is the business key | Halt pipeline |
| `invalid_count(price) — valid min: 0` | FAIL | Negative prices break revenue | Halt pipeline |
| `invalid_count(sku) — valid regex` | WARN | Format violation = data quality debt | Log + alert |

---

## Table: orders

**Owner:** Fulfilment Team  
**Contract:** `soda/data-contract-orders.yml`  
**Checks file:** `soda/checks_orders.yml`

| Check | Threshold | Rationale | Action on FAIL |
|---|---|---|---|
| `row_count > 0` | FAIL | Pipeline should not produce empty orders | Halt pipeline |
| `missing_count(id) = 0` | FAIL | PK must never be NULL | Halt pipeline |
| `duplicate_count(id) = 0` | FAIL | PK must be unique | Halt pipeline |
| `missing_percent(amount) warn >0.5, fail >1.0` | WARN/FAIL | Revenue accuracy SLA | Warn: log, Fail: halt |
| `invalid_count(status)` | FAIL | Only valid lifecycle states allowed | Halt pipeline |
| `shipped_before_ordered` (SQL check) | FAIL | Logical constraint — never valid | Halt pipeline |

---

## Table: order_items

**Owner:** Fulfilment Team  
**Checks file:** `soda/checks_order_items.yml`

| Check | Threshold | Rationale | Action on FAIL |
|---|---|---|---|
| `row_count > 0` | FAIL | Orders must have items | Halt pipeline |
| `missing_count(order_id) = 0` | FAIL | FK must not be NULL | Halt pipeline |
| `invalid_count(quantity) — valid min: 1` | FAIL | Zero-quantity items are invalid | Halt pipeline |
| `invalid_count(unit_price) — valid min: 0` | FAIL | Negative prices break billing | Halt pipeline |

---

## Escalation Contacts

| Severity | Contact | Response SLA |
|---|---|---|
| FAIL (blocking) | On-call engineer (PagerDuty) | 15 minutes |
| WARN (non-blocking) | Table owner (Slack) | Next business day |

---

*Framework version 1.0 — Soda Core for Data Quality*
"""

quality_md_path = tmpdir / "QUALITY.md"
quality_md_path.write_text(quality_md)

print("QUALITY.md template written to:", quality_md_path)
print()
print(quality_md)

### What just happened?

- **QUALITY.md** is the human-readable documentation of your framework — it tells every engineer what each check does, why the threshold was chosen, and who to contact when it fails.
- Store it in the root of your data platform repo and keep it in sync with the SodaCL checks files.
- The escalation table ensures that when Soda sends an alert, there is a clear owner and SLA for every table.

## Step 9 · Full Pipeline Scan Report

In [ ]:
# Run all four tables in a single combined scan
scan_full = Scan()
scan_full.set_data_source_name("ecommerce")
scan_full.add_configuration_yaml_file(str(config_path))
scan_full.add_sodacl_yaml_file(str(checks_path))  # the combined checks_all.yml
scan_full.execute()

print("=" * 60)
print("FULL PIPELINE SCAN REPORT")
print("=" * 60)
print(scan_full.get_logs_text())

exit_code = scan_full.get_exit_code()
overall = {0: "✅ ALL PASS", 1: "⚠️  WARNINGS", 2: "❌ FAILURES"}.get(exit_code, "UNKNOWN")
print(f"\nOverall: {overall} (exit code {exit_code})")

## Challenge

```python
# Challenge: Extend the framework with a custom SQL referential integrity check.
#
# 1. Add a new checks file: checks_referential.yml
#    checks for orders:
#      - failed rows:
#          fail condition: >
#            SELECT o.id FROM orders o
#            LEFT JOIN customers c ON o.customer_id = c.id
#            WHERE c.id IS NULL
#          name: orphan_orders_no_customer
#          fail: when > 0
#
# 2. Run this check — it should FAIL because order 5 references customer_id=9
#    which does not exist in the customers table.
#
# 3. Add the check to the orchestrator as a non-blocking stage.
#
# 4. Add a row to the QUALITY.md table for this check:
#    | `orphan_orders_no_customer` | FAIL | Referential integrity | Halt pipeline |

# Your solution here
```

---
## Day 10 key concepts recap

| Concept | What you built |
|---|---|
| Multi-table database | 4-table e-commerce schema with injected quality issues |
| Comprehensive checks YAML | row_count, missing, duplicates, validity, schema, SQL checks |
| `DataQualityOrchestrator` | Dependency-ordered scan execution; halts on upstream FAIL |
| Per-table checks files | Modular checks files: one per table, composed in the orchestrator |
| Data contract | Formal YAML contract for `orders`: schema + SLA + thresholds |
| Schema drift detection | `schema:` check catches breaking column renames in one scan |
| QUALITY.md | Human-readable framework documentation with per-check rationale |
| Exit code gating | `get_exit_code() == 2` blocks downstream pipeline steps |

> **Final tip:** A data quality framework is a living document. Start with the four critical checks per table (row_count, PK missing, PK duplicate, one domain rule), ship it, and iterate. The best DQ framework is the one your team actually runs on every pipeline execution.

---
## Congratulations — Soda Core for Data Quality Complete!

You've covered the full arc: architecture → connectors → checks → thresholds → custom SQL → pipeline integration → Cloud → schema evolution → TDDD → capstone.

Mark Day 10 complete in your [tracker](../index.html).